# Encapsulamiento

## Aplicación en el Modelo Bancario

In [4]:
from abc import ABC, abstractmethod
from datetime import datetime
from typing import List

class Cuenta(ABC):

    def __init__(self, numero_cuenta: str, titular: str, saldo_inicial: float = 0):
        self.numero_cuenta = numero_cuenta  # Público
        self.titular = titular              # Público
        self.__saldo = saldo_inicial
        self.fecha_creacion = datetime.now() # Público
        self.movimientos = []               # Público

    def obtener_saldo(self) -> float:
        """Método PÚBLICO para consultar el saldo"""
        return self.__saldo

    def depositar(self, cantidad: float) -> bool:
        """Operación controlada: validamos antes de modificar"""
        if cantidad <= 0:
            print(f"Cantidad debe ser positiva")
            return False

        self.__saldo += cantidad
        self.movimientos.append({
            "tipo": "deposito",
            "cantidad": cantidad,
            "saldo_resultado": self.__saldo,
            "fecha": datetime.now()
        })
        print(f"✓ Depósito de ${cantidad:.2f} realizado exitosamente")
        return True

    def retirar(self, cantidad: float) -> bool:
        if cantidad <= 0:
            print(f"Cantidad debe ser positiva")
            return False

        if cantidad > self.__saldo:
            print(f"Saldo insuficiente. Disponible: ${self.__saldo:.2f}")
            return False

        self.__saldo -= cantidad
        self.movimientos.append({
            "tipo": "retiro",
            "cantidad": cantidad,
            "saldo_resultado": self.__saldo,
            "fecha": datetime.now()
        })
        print(f"✓ Retiro de ${cantidad:.2f} realizado exitosamente")
        return True

    @abstractmethod
    def procesar_comisiones(self):
        pass

In [8]:

class CuentaAhorros(Cuenta):
    def __init__(self, numero_cuenta: str, titular: str, saldo_inicial: float = 0, tasa_interes: float = 0.05):
        super().__init__(numero_cuenta, titular, saldo_inicial)
        self._tasa_interes = tasa_interes

    def procesar_comisiones(self):
        interes = self.__saldo * (self._tasa_interes / 12)
        self.__saldo += interes
        print(f"Interés de ${interes:.2f} acreditado")

cuenta = CuentaAhorros("AH-001", "Leidis López", 5000, 0.05)

print(f"Consultamos saldo mediante método: ${cuenta.obtener_saldo():.2f}")
cuenta.depositar(-500)
cuenta.retirar(10000)
cuenta.depositar(1000)  # Se acepta
print(f"Saldo actual: ${cuenta.obtener_saldo():.2f}")


for i, mov in enumerate(cuenta.movimientos, 1):
    if mov['tipo'] == 'deposito':
        print(f"  {i}. Depósito de ${mov['cantidad']:.2f} - Saldo: ${mov['saldo_resultado']:.2f}")
    elif mov['tipo'] == 'retiro':
        print(f"  {i}. Retiro de ${mov['cantidad']:.2f} - Saldo: ${mov['saldo_resultado']:.2f}")

print(f"  Saldo = {cuenta.obtener_saldo():.2f}")


Consultamos saldo mediante método: $5000.00
Cantidad debe ser positiva
Saldo insuficiente. Disponible: $5000.00
✓ Depósito de $1000.00 realizado exitosamente
Saldo actual: $6000.00
  1. Depósito de $1000.00 - Saldo: $6000.00
  Saldo = 6000.00


## Atributos Protegidos Específicos por Subclase

In [9]:

class CuentaCorriente(Cuenta):
    def __init__(self, numero_cuenta: str, titular: str, saldo_inicial: float = 0,
                 comision_mensual: float = 10, limite_sobregiro: float = 500):
        super().__init__(numero_cuenta, titular, saldo_inicial)
        self._comision_mensual = comision_mensual  # Protegido
        self._limite_sobregiro = limite_sobregiro  # Protegido

    def obtener_comision_mensual(self) -> float:
        return self._comision_mensual

    def obtener_limite_sobregiro(self) -> float:
        return self._limite_sobregiro

    def establecer_comision_mensual(self, nueva_comision: float):
        if nueva_comision < 0:
            print("La comisión no puede ser negativa")
            return
        self._comision_mensual = nueva_comision
        print(f"✓ Comisión actualizada a ${nueva_comision:.2f}")

    def procesar_comisiones(self):
        if self.__saldo >= self._comision_mensual:
            self.__saldo -= self._comision_mensual
            print(f"Comisión de ${self._comision_mensual:.2f} cobrada")

cuenta_cc = CuentaCorriente("CC-001", "Leidis López", 1000, 15, 500)

print(f"Comisión mensual: ${cuenta_cc.obtener_comision_mensual():.2f}")
print(f"Límite de sobregiro: ${cuenta_cc.obtener_limite_sobregiro():.2f}")

print(f"  Antes: ${cuenta_cc._comision_mensual:.2f}")
cuenta_cc._comision_mensual = -100
print(f"  Después: ${cuenta_cc._comision_mensual:.2f} ¡Comisión negativa!")

cuenta_cc.establecer_comision_mensual(-100)
print(f"  Comisión actual: ${cuenta_cc.obtener_comision_mensual():.2f}")
cuenta_cc.establecer_comision_mensual(20)

Comisión mensual: $15.00
Límite de sobregiro: $500.00
  Antes: $15.00
  Después: $-100.00 ¡Comisión negativa!
La comisión no puede ser negativa
  Comisión actual: $-100.00
✓ Comisión actualizada a $20.00
